# Inverse DC-OPF: end-to-end Colab pipeline

Runs every experiment used in the EE364B final paper and prints the numbers needed for Table I and the noise / capacity / PJM-like sweeps.

**Recommended runtime:** CPU is enough; everything finishes in 15-30 min total. GPU does not help (the bottleneck is the CVXPY/SCS forward solve, which runs on CPU regardless).

## How to use
1. Open this notebook in Colab.
2. Run the **Setup** cell. It will install the package and may force a kernel restart (Colab prints "Your session crashed for an unknown reason" — that is normal, just keep going).
3. After restart, run **all cells from "Quick verification" onwards**.
4. The final cell writes a single markdown summary with every published number.

## 0. Setup

Clones the repo and installs the package in editable mode. Pins from `pyproject.toml` are honored, so you get exactly the versions that produced the paper's numbers.

If the repository is private, replace the clone URL with `https://<TOKEN>@github.com/carlo-scr/ee364bfinal.git`.

In [ ]:
import os, sys, subprocess, shutil
REPO_URL = "https://github.com/carlo-scr/ee364bfinal.git"
REPO_DIR = "/content/ee364bfinal"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
print("contents:", sorted(os.listdir(".")))

In [ ]:
# Install dependencies then the package.
# Upgrades pip/setuptools first so the src-layout editable install works.
import subprocess, sys

def pip(*args, quiet=True):
    cmd = [sys.executable, "-m", "pip", "install"] + (["-q"] if quiet else []) + list(args)
    r = subprocess.run(cmd)
    if r.returncode != 0:
        raise RuntimeError(f"pip failed (exit {r.returncode}): {' '.join(cmd)}")

# Ensure a recent pip + setuptools so editable src-layout installs work.
pip("--upgrade", "pip", "setuptools", "wheel")

# Core scientific stack.
pip("numpy>=2.0", "scipy>=1.13", "scikit-learn>=1.5",
    "matplotlib>=3.8", "pandas>=2.2", "seaborn>=0.13", "pyyaml>=6.0")

# Convex optimization stack.
pip("cvxpy>=1.5", "scs>=3.2", "cvxpylayers>=0.1.6")

# Install the package itself (no deps, they are already installed above).
# quiet=False so any build error is visible.
pip("-e", ".", "--no-deps", quiet=False)
print("install OK")


In [ ]:
# If torch / numpy were upgraded above, Colab needs a kernel restart for the
# new versions to be picked up.  This cell triggers it cleanly; just run all
# subsequent cells once the kernel comes back.
import os, IPython
if os.environ.get("INVERSE_OPF_RESTARTED") != "1":
    os.environ["INVERSE_OPF_RESTARTED"] = "1"
    IPython.Application.instance().kernel.do_shutdown(True)

## 1. Quick verification

Runs the unit test suite (~15 s) to confirm the install is healthy.

In [ ]:
import os, subprocess
os.chdir("/content/ee364bfinal")
res = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v", "--tb=short",
     "-W", "ignore::DeprecationWarning", "-W", "ignore::UserWarning"],
    capture_output=True, text=True,
)
print(res.stdout[-3000:])
if res.returncode != 0:
    print("STDERR:", res.stderr[-2000:])
    raise RuntimeError("pytest failed")

In [ ]:
# Smoke run of the unified CLI: 2 seeds, tiny network.  Should print one
# summary table in ~30 s and prove the cache / test-eval path works.
import subprocess
res = subprocess.run(
    ["python", "scripts/run.py", "--config", "configs/smoke.yaml",
     "--output-root", "/content/outputs_smoke"],
    capture_output=True, text=True,
)
print(res.stdout[-3000:])
assert res.returncode == 0, res.stderr[-2000:]

## 2. Headline methods comparison (Table I in the paper)

Five seeds, held-out test set, six methods. Prints per-seed timings and the bootstrap-CI summary.

**Runtime:** ~6-10 min on Colab CPU.

In [ ]:
import subprocess, time
t0 = time.time()
res = subprocess.run(
    ["python", "scripts/run.py",
     "--config", "configs/methods_comparison_5seed.yaml",
     "--output-root", "/content/outputs"],
    capture_output=True, text=True,
)
print(res.stdout[-6000:])
assert res.returncode == 0, res.stderr[-2000:]
print(f"\nelapsed: {time.time()-t0:.1f} s")

In [ ]:
import pandas as pd
summary = pd.read_csv("/content/outputs/methods_comparison_5seed/summary.csv")
# Headline columns only.
cols = ["method",
        "val_nrmse_clean_mean", "val_nrmse_clean_ci_lo", "val_nrmse_clean_ci_hi",
        "test_nrmse_clean_mean", "test_nrmse_clean_ci_lo", "test_nrmse_clean_ci_hi",
        "f_cos_mean", "f_cos_ci_lo", "f_cos_ci_hi",
        "f_merit_acc_mean", "f_merit_acc_ci_lo", "f_merit_acc_ci_hi",
        "gmax_cos_mean", "gmax_cos_ci_lo", "gmax_cos_ci_hi"]
cols = [c for c in cols if c in summary.columns]
display = summary[cols].copy()

def _fmt_with_ci(row, base):
    m = row[f"{base}_mean"]
    lo = row.get(f"{base}_ci_lo", float("nan"))
    hi = row.get(f"{base}_ci_hi", float("nan"))
    return f"{m:.3f} [{lo:.3f}, {hi:.3f}]"

pretty = pd.DataFrame({"method": summary["method"]})
for base in ["val_nrmse_clean", "test_nrmse_clean", "f_cos", "f_merit_acc", "gmax_cos"]:
    if f"{base}_mean" in summary.columns:
        pretty[base] = [_fmt_with_ci(r, base) for _, r in summary.iterrows()]
print(pretty.to_string(index=False))

## 3. Tight-capacity variant

Same five seeds, but `gmax_scale=[1.02, 1.10]` so more generators bind. Tests whether the differentiable estimator still recovers capacities under sharper constraints.

**Runtime:** ~8-12 min.

In [ ]:
import subprocess, time
t0 = time.time()
res = subprocess.run(
    ["python", "scripts/run.py",
     "--config", "configs/methods_comparison_tightcaps.yaml",
     "--output-root", "/content/outputs"],
    capture_output=True, text=True,
)
print(res.stdout[-6000:])
assert res.returncode == 0, res.stderr[-2000:]
print(f"\nelapsed: {time.time()-t0:.1f} s")

In [ ]:
import pandas as pd
summary_tight = pd.read_csv("/content/outputs/methods_comparison_tightcaps/summary.csv")
pretty = pd.DataFrame({"method": summary_tight["method"]})
for base in ["val_nrmse_clean", "test_nrmse_clean", "f_cos", "f_merit_acc", "gmax_cos"]:
    if f"{base}_mean" in summary_tight.columns:
        pretty[base] = [
            f"{r[f'{base}_mean']:.3f} [{r.get(f'{base}_ci_lo', float('nan')):.3f}, {r.get(f'{base}_ci_hi', float('nan')):.3f}]"
            for _, r in summary_tight.iterrows()
        ]
print(pretty.to_string(index=False))

## 3b. PJM-like 8-fuel stratified recovery

Real-data-flavoured synthetic benchmark: one bus, 8 fuels (nuclear / hydro / wind / solar / coal / CCGT / oil / peaker) with hour-of-day strata.  Reports merit-order accuracy and stratified-table cosine.

In [ ]:
import subprocess, time
import pandas as pd

t0 = time.time()
subprocess.check_call([
    "python", "scripts/run.py",
    "--config", "configs/pjm_like.yaml",
    "--output-root", "/content/outputs",
], cwd="/content/ee364bfinal")
print(f"PJM-like done in {time.time()-t0:.1f}s")

summary_pjm = pd.read_csv("/content/outputs/pjm_like_3seed/summary.csv")
summary_pjm


## 3c. Missing-data ablation (50% of training samples dropped)

Same headline pipeline but each seed only sees half of the original training rows; tests robustness to reduced observation count.

In [ ]:
import subprocess, time
import pandas as pd

t0 = time.time()
subprocess.check_call([
    "python", "scripts/run.py",
    "--config", "configs/methods_comparison_missing50.yaml",
    "--output-root", "/content/outputs",
], cwd="/content/ee364bfinal")
print(f"missing-data done in {time.time()-t0:.1f}s")

summary_miss = pd.read_csv("/content/outputs/methods_comparison_missing50/summary.csv")
summary_miss


## 3d. Sensitivity / ablation suite (Sections 2-6)

Runs the eight new registry experiments in sequence: epsilon sweep,
warm-start, k-means strata recovery, congestion analysis, capacity
screening, per-method timing, marginal emission factors, and
counterfactual CO2 dispatch.


In [ ]:
import subprocess, time
import pandas as pd

extra_configs = [
    ("timing",         "configs/timing.yaml"),
    ("warmstart",      "configs/warmstart.yaml"),
    ("kmeans_strata",  "configs/kmeans_strata.yaml"),
    ("congestion",     "configs/congestion.yaml"),
    ("screening",      "configs/screening.yaml"),
    ("mef",            "configs/mef.yaml"),
    ("counterfactual", "configs/counterfactual.yaml"),
    ("epsilon_sweep",  "configs/epsilon_sweep.yaml"),
]
for name, cfg in extra_configs:
    print(f"\n=== {name} ===")
    t0 = time.time()
    subprocess.check_call(["python","scripts/run.py","--config",cfg,
                           "--output-root","/content/outputs"])
    print(f"  elapsed {time.time()-t0:.1f}s")


## 4. Legacy single-script experiments

These pre-date the unified CLI but still produce numbers used in the paper. They write into `outputs/<name>/` exactly as before.

* `run_noise_sweep.py`  — RMSE vs. observation noise level.
* `run_recovery_curve.py` — RMSE vs. number of training samples.
* `run_ablation.py` — Tikhonov regularization ablation.
* `run_identifiability.py` — per-generator identifiability scores.
* `run_pjm_like.py` — single-bus, 8-fuel PJM-like merit order recovery.
* `run_mef_analysis.py` — marginal emissions factor estimation.

Each is independently runnable; pick the ones you need. The cell below runs the fastest two as a sanity check.

In [ ]:
import subprocess, time
for script, args in [
    ("scripts/run_noise_sweep.py",      ["--seeds", "0,1", "--noises", "0.1,0.5,1.0"]),
    ("scripts/run_recovery_curve.py",   ["--config", "configs/synthetic_small.yaml",
                                          "--seeds", "0,1", "--sizes", "50,100,200"]),
]:
    print(f"\n=== {script} ===")
    t0 = time.time()
    res = subprocess.run(["python", script] + args, capture_output=True, text=True)
    print(res.stdout[-2500:])
    if res.returncode != 0:
        print("STDERR:", res.stderr[-1500:])
    print(f"elapsed: {time.time()-t0:.1f} s")

## 5. Final consolidated summary

Collects every CSV produced above into one markdown report. Save this — it contains every number you need to plug into the paper or compare against later.

In [ ]:
import glob, os, pandas as pd, datetime, json

def _df_to_markdown(df, max_rows=20):
    df = df.head(max_rows).copy()
    for c in df.columns:
        if df[c].dtype.kind == "f":
            df[c] = df[c].map(lambda v: f"{v:.4f}" if pd.notna(v) else "nan")
    return df.to_markdown(index=False)

sections = ["# Inverse DC-OPF results\n",
            f"Generated {datetime.datetime.utcnow().isoformat(timespec='seconds')}Z\n"]

for label, path in [
    ("Headline (5 seeds, default caps)", "/content/outputs/methods_comparison_5seed/summary.csv"),
    ("Tight capacity budget",            "/content/outputs/methods_comparison_tightcaps/summary.csv"),
    ("PJM-like 8-fuel stack",            "/content/outputs/pjm_like_3seed/summary.csv"),
    ("Missing-data (50%% dropped)",       "/content/outputs/methods_comparison_missing50/summary.csv"),
    ("Eps sweep (Sec. 2)",                "/content/outputs/epsilon_sweep/summary.csv"),
    ("Warm-start (Sec. 3)",               "/content/outputs/warmstart/summary.csv"),
    ("K-means strata (Sec. 3)",           "/content/outputs/kmeans_strata/summary.csv"),
    ("Congestion (Sec. 2)",               "/content/outputs/congestion/summary.csv"),
    ("Capacity screening (Sec. 4)",       "/content/outputs/screening/summary.csv"),
    ("Per-method timing (Sec. 4)",        "/content/outputs/timing/summary.csv"),
    ("Marginal emission factor (Sec. 6)", "/content/outputs/mef/summary.csv"),
    ("Counterfactual CO2 (Sec. 6)",       "/content/outputs/counterfactual/summary.csv"),
]:
    if os.path.exists(path):
        df = pd.read_csv(path)
        sections.append(f"## {label}\n\n{_df_to_markdown(df)}\n")

# Pick up legacy outputs.
for csv in sorted(glob.glob("/content/ee364bfinal/outputs/**/*.csv", recursive=True)):
    rel = csv.replace("/content/ee364bfinal/", "")
    try:
        df = pd.read_csv(csv)
    except Exception:
        continue
    sections.append(f"## {rel}\n\n{_df_to_markdown(df)}\n")

report = "\n".join(sections)
out_path = "/content/inverse_opf_report.md"
with open(out_path, "w") as f:
    f.write(report)
print(f"wrote {out_path} ({len(report)} chars)")
print("\n--- preview ---\n")
print(report[:4000])

In [ ]:
# Download the report to your local machine (Colab helper).
try:
    from google.colab import files
    files.download("/content/inverse_opf_report.md")
except Exception as e:
    print("not in Colab or download blocked:", e)